In [ ]:
import re
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse.csgraph import dijkstra
from pathlib import Path
from joblib import Parallel, delayed

In [ ]:
# ===================================
# 0. CONSTANTS
# ===================================
EPSILON = 1e-10

# Metadata columns of the feature tables (everything else is a graph feature)
META_COLS = {"Patient", "Class", "SeizureID", "Block", "Band", "Epoch", "split"}

In [ ]:
# ===================================
# 1. FEATURE COMPUTATION FUNCTIONS
# ===================================
def _to_distance(W):
    """Convert connectivity weights (E, C, C) to distances: 1/W for W > 0, inf otherwise; zero diagonal."""
    D = np.where(W > 0, 1.0 / np.maximum(W, EPSILON), np.inf)
    idx = np.arange(W.shape[-1])
    D[:, idx, idx] = 0.0
    return D

def _clustering_batch(W):
    wmax = W.max(axis=(1, 2), keepdims=True)
    A = np.cbrt(W / np.where(wmax > 0, wmax, 1.0))
    tri = np.einsum('eij,ejk,eki->ei', A, A, A)
    k = (W > 0).sum(-1).astype(float)
    denom = k * (k - 1)
    return np.divide(tri, denom, out=np.zeros_like(tri), where=denom > 0)

def _betweenness_batch(Wt, D, rtol=1e-9):
    """
    Wt: (E, C, C) thresholded PLI; D: (E, C, C) shortest-path distances of Wt.
    Normalised like nx.betweenness_centrality(normalized=True) for undirected graphs.
    """
    E, C, _ = Wt.shape
    Wd = _to_distance(Wt)
    idx = np.arange(C)
    Wd[:, idx, idx] = np.inf
    eye = np.eye(C, dtype=bool)

    # edge (k, t) lies on a shortest path s -> t
    tot = D[:, :, :, None] + Wd[:, None, :, :]
    M = (np.isclose(tot, D[:, :, None, :], rtol=rtol, atol=0)
         & np.isfinite(tot)).astype(float)

    # number of shortest paths sigma[e, s, t]
    I = np.broadcast_to(np.eye(C), (E, C, C))
    sigma = I.copy()
    for _ in range(C - 1):
        new = I + np.einsum('esk,eskt->est', sigma, M)
        if np.array_equal(new, sigma):
            break
        sigma = new

    # v lies on a shortest path s -> t (v != s, v != t, s != t)
    sv = D[:, :, :, None] + D[:, None, :, :]
    on = np.isclose(sv, D[:, :, None, :], rtol=rtol, atol=0) & np.isfinite(sv)
    on &= ~eye[None, :, :, None]
    on &= ~eye[None, None, :, :]
    on &= ~eye[None, :, None, :]

    ratio = (sigma[:, :, :, None] * sigma[:, None, :, :]
             / np.where(sigma[:, :, None, :] > 0, sigma[:, :, None, :], 1.0))
    return (ratio * on).sum(axis=(1, 3)) / ((C - 1) * (C - 2))


def _dijkstra_batch(D):
    return np.stack([dijkstra(d, directed=False) for d in D])

def average_shortest_path(W):
    sp = _dijkstra_batch(_to_distance(W))            # (E, C, C)
    C = sp.shape[-1]
    off_diag = ~np.eye(C, dtype=bool)

    d = sp[:, off_diag]                              # (E, C*(C-1)) pairs i != j
    connected = np.isfinite(d).all(axis=1)           # (E,)
    return np.where(connected, d.mean(axis=1), np.nan)

def _eigen_centrality_batch(W):
    _, vecs = np.linalg.eigh(W)
    v = np.abs(vecs[..., -1])
    nrm = np.linalg.norm(v, axis=-1, keepdims=True)
    return v / np.where(nrm > 0, nrm, 1.0)

def _features_chunk(W, ratio_thres):
    C = W.shape[-1]

    # --- Original graph ---
    cluster  = _clustering_batch(W)
    strength = W.sum(-1)
    eigen    = _eigen_centrality_batch(W)
    avg_path = average_shortest_path(W)

    # --- Thresholded graph ---
    thres = ratio_thres * W.max(axis=(1, 2), keepdims=True)
    Wt = np.where(W > thres, W, 0.0)
    density = np.triu(Wt > 0, k=1).sum(axis=(1, 2)) / (C * (C - 1) / 2)
    Dt = _dijkstra_batch(_to_distance(Wt))
    inv = np.where(np.isfinite(Dt) & (Dt > 0), 1.0 / np.where(Dt > 0, Dt, 1.0), 0.0)
    wge = inv.sum(axis=(1, 2)) / (C * (C - 1))
    between = _betweenness_batch(Wt, Dt)

    cols = {}
    for name, arr in [('Clustering_Coeff', cluster), ('Strength', strength),
                      ('Eigen_Centrality', eigen), ('Betweenness_Centrality', between)]:
        for i in range(C):
            cols[f"Node_{i}_{name}"] = arr[:, i]
    cols['Density'] = density
    cols['Weighted_Global_Efficiency'] = wge
    cols['Characteristic_Path_Length'] = avg_path
    return pd.DataFrame(cols)


def compute_features_batch(pli, ratio_thres, chunk=256):
    """pli: (E, C, C) symmetric. Returns a DataFrame with one row per epoch."""
    W = np.asarray(pli, dtype=float).copy()
    idx = np.arange(W.shape[-1])
    W[:, idx, idx] = 0.0
    parts = [_features_chunk(W[s:s + chunk], ratio_thres)
             for s in range(0, len(W), chunk)]
    return pd.concat(parts, ignore_index=True)

In [ ]:
# ==========================================================================
# 2. FEATURE EXTRACTION (TRAINING & TESTING)
# ==========================================================================
def _num(s):
    """First integer in a folder name: 'seizure_00' -> 0, '1' -> 1."""
    return int(re.search(r'\d+', s).group())

def extract_features_training(root, ratio_thres=0.5, patients=None, bands=None,
                              chunk=256, saved_closest_first=False):
    """
    root: folder .../Connectivity_matrics/Training/PLI_matrix
      preictal:   <patient>/preictal/<seizure>/<band>.npy
      interictal: <patient>/interictal/seizure_XX/<segment>/<band>.npy
    patients: list of patients (e.g. ['chb01']); None = all
    bands:    list of bands (e.g. ['delta', 'alpha']); None = all
    saved_closest_first: True if, inside the .npy files, the epoch closest to the
                         seizure comes FIRST (the order is reversed so that the
                         largest Epoch = closest to the seizure).

    Returns a DataFrame: Patient, Class, SeizureID, Band, Epoch + features
    (empty DataFrame if no .npy file is found).
    SeizureID is the number in the folder name (seizure_00 -> 0, preictal/1 -> 1).
    """
    root = Path(root)
    pdirs = ([root / p for p in patients] if patients
             else sorted(d for d in root.iterdir() if d.is_dir()))

    # (patient, class, seizure, band) -> [(segment, path), ...]
    groups = defaultdict(list)
    for pdir in pdirs:
        for f in (pdir / 'preictal').glob('*/*.npy'):
            groups[(pdir.name, 'preictal', _num(f.parent.name), f.stem)].append((0, f))
        for f in (pdir / 'interictal').glob('*/*/*.npy'):
            groups[(pdir.name, 'interictal', _num(f.parent.parent.name), f.stem)] \
                .append((_num(f.parent.name), f))

    frames = []
    for (patient, cls, seizure, band), files in sorted(groups.items()):
        if bands and band not in bands:
            continue
        files.sort(key=lambda x: x[0])                    # concatenate in segment order

        mats = []
        for _, f in files:
            a = np.load(f)
            if a.ndim != 3 or a.shape[1] != a.shape[2]:
                raise ValueError(f"{f}: expected shape (E, C, C), got {a.shape}")
            mats.append(a)
        W = np.concatenate(mats, axis=0)

        epoch = np.arange(len(W))                         # 0 .. E-1, continuous across segments
        if saved_closest_first:
            epoch = epoch[::-1]                           # largest epoch = closest to the seizure

        feats = compute_features_batch(W, ratio_thres, chunk=chunk)
        meta = pd.DataFrame({
            'Patient': patient, 'Class': cls, 'SeizureID': seizure,
            'Band': band, 'Epoch': epoch,
        })
        frames.append(pd.concat([meta, feats], axis=1))

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

def extract_features_testing(root, ratio_thres=0.5, patients=None, bands=None,
                             chunk=256, saved_closest_first=False):
    """
    root: folder .../Connectivity_matrics/Testing/wPLI_matrix
      layout: <patient>/seizure_XX/block_YY/<number>/<band>.npy
    patients: list of patients (e.g. ['chb04']); None = all
    bands:    list of bands (e.g. ['theta']); None = all
    saved_closest_first: True if the time order in the data runs from the seizure
                         outwards (the order is reversed so that the largest
                         Epoch = closest to the seizure).

    Returns a DataFrame: Patient, SeizureID, Block, Band, Epoch + features
    (empty DataFrame if no .npy file is found).
    Epoch is numbered continuously per (patient, seizure, band), ordered by
    (block, sub-folder number).
    """
    root = Path(root)
    pdirs = ([root / p for p in patients] if patients
             else sorted(d for d in root.iterdir() if d.is_dir()))

    # (patient, seizure, band) -> [(block, segment, path), ...]
    groups = defaultdict(list)
    for pdir in pdirs:
        for f in pdir.glob('seizure_*/block_*/*/*.npy'):
            seg_dir, block_dir, seizure_dir = f.parent, f.parent.parent, f.parent.parent.parent
            groups[(pdir.name, _num(seizure_dir.name), f.stem)] \
                .append((_num(block_dir.name), _num(seg_dir.name), f))

    frames = []
    for (patient, seizure, band), files in sorted(groups.items()):
        if bands and band not in bands:
            continue
        files.sort(key=lambda x: (x[0], x[1]))            # by block, then by sub-folder number

        mats, blocks = [], []
        for block, _, f in files:
            a = np.load(f)
            if a.ndim != 3 or a.shape[1] != a.shape[2]:
                raise ValueError(f"{f}: expected shape (E, C, C), got {a.shape}")
            mats.append(a)
            blocks.append(np.full(len(a), block))
        W = np.concatenate(mats, axis=0)

        epoch = np.arange(len(W))
        if saved_closest_first:
            epoch = epoch[::-1]

        feats = compute_features_batch(W, ratio_thres, chunk=chunk)
        meta = pd.DataFrame({
            'Patient': patient, 'SeizureID': seizure,
            'Block': np.concatenate(blocks),
            'Band': band, 'Epoch': epoch,
        })
        frames.append(pd.concat([meta, feats], axis=1))

    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

In [ ]:
# ==========================================================================
# 3. SAVE FEATURES
# ==========================================================================
def save_features(out_root, df_train=None, df_test=None):
    """
    One folder per patient:
      <out_root>/<patient>/preictal.csv                   (training preictal, all seizures)
      <out_root>/<patient>/seizure_XX_interictal.csv      (training interictal, all segments of seizure XX)
      <out_root>/<patient>/seizure_XX_block_YY.csv        (testing, block YY of seizure XX)

    df_train: output of extract_features_training (columns Class, SeizureID, ...)
    df_test:  output of extract_features_testing  (columns SeizureID, Block, ...)
    """
    out_root = Path(out_root)

    if df_train is not None and not df_train.empty:
        for patient, dp in df_train.groupby('Patient'):
            pdir = out_root / patient
            pdir.mkdir(parents=True, exist_ok=True)

            pre = dp[dp['Class'] == 'preictal']
            if len(pre):
                pre.to_csv(pdir / 'preictal.csv', index=False)

            inter = dp[dp['Class'] == 'interictal']
            for seizure, d in inter.groupby('SeizureID'):
                d.to_csv(pdir / f'seizure_{seizure:02d}_interictal.csv', index=False)

    if df_test is not None and not df_test.empty:
        for (patient, seizure, block), d in df_test.groupby(['Patient', 'SeizureID', 'Block']):
            pdir = out_root / patient
            pdir.mkdir(parents=True, exist_ok=True)
            d.to_csv(pdir / f'seizure_{seizure:02d}_block_{block:02d}.csv', index=False)

In [ ]:
# ==========================================================================
# 4. RUN ALL PATIENTS IN PARALLEL + MERGE TRAIN / TEST
# ==========================================================================
def _run_patient(p, train_root, test_root, out_root, ratio_thres, bands):
    """Extract and save training/testing features for one patient."""
    df_train = None
    df_test = None
    if (train_root / p).exists():
        df_train = extract_features_training(
            train_root,
            ratio_thres,
            patients=[p],
            bands=bands
        )
    if (test_root / p).exists():
        df_test = extract_features_testing(
            test_root,
            ratio_thres,
            patients=[p],
            bands=bands
        )
    # Save this patient's results
    save_features(out_root, df_train, df_test)

    return p, df_train, df_test


def run_all(train_root,test_root,out_root,ratio_thres=0.5,bands=None,patients=None,n_jobs=-1):
    train_root = Path(train_root)
    test_root = Path(test_root)
    out_root = Path(out_root)

    # Automatically discover patients
    if patients is None:
        names = {d.name for root in (train_root, test_root) if root.exists() for d in root.iterdir() if d.is_dir()}
        patients = sorted(names)

    print(f"Total patients: {len(patients)}")
    print(f"Parallel workers: {n_jobs}")
    print("Starting feature extraction...")

    # Process patients in parallel
    results = Parallel(n_jobs=n_jobs, verbose=10)(
        delayed(_run_patient)(p,train_root,test_root,out_root,ratio_thres,bands) for p in patients)

    # Report completion
    for p, df_train, df_test in results:
        n_train = 0 if df_train is None else len(df_train)
        n_test = 0 if df_test is None else len(df_test)

        print(
            f"Done {p} | "
            f"Training rows: {n_train:,} | "
            f"Testing rows: {n_test:,}"
        )

    print("All patients completed.")

# ==========================================================================
# 5. RUN
# ==========================================================================

run_all(
    r"D:\Downloads\Compressed\Connectivity_matrics\Training\wPLI_matrix",
    r"D:\Downloads\Compressed\Connectivity_matrics\Testing\wPLI_matrix",
    r"D:\Downloads\Compressed\Graph-based_Feature\wPLI_features_0.6",
    ratio_thres=0.6,
    bands=None,
    n_jobs=-1
)